# 2 · ML: una columna nueva para clasificar puestos

Entrenamos un clasificador de texto pequeño y legible con ejemplos de enseñanza etiquetados por una persona. El split `train/test` está fijado en el fixture, así que la evaluación no cambia entre ejecuciones. Son títulos sintéticos, no respuestas etiquetadas por un modelo; cinco clases y unas decenas de ejemplos solo ilustran el flujo, no dan precisión de producción.

Al final registramos la función Python como `clasificar_persona_ml(puesto_texto)` dentro de la sesión Spark y la invocamos desde SQL contra títulos sintéticos y, si ya existe, contra la encuesta C2. El texto de participantes no se muestra; la consulta final devuelve solo conteos por clase.

**Compute:** Serverless notebook con `scikit-learn==1.6.1` y `matplotlib==3.10.0`. En el panel *Environment*, agrega `scikit-learn==1.6.1` antes de ejecutar; el runner del repo ya lo instala en la tarea.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay
from sklearn.pipeline import Pipeline

FIXTURE = "/Workspace/Shared/AI4Data/Clase2/training/persona_labels.csv"
CATALOG = "workspace"
SCHEMA = "ai4data"
DIM = f"{CATALOG}.{SCHEMA}.dim_participante"

labels = pd.read_csv(FIXTURE)
train = labels.loc[labels["split"] == "train"].copy()
holdout = labels.loc[labels["split"] == "test"].copy()
classes = ["ejecutivo", "manager", "practitioner", "estudiante", "otro"]
assert set(train["persona_esperada"]) == set(classes)
assert set(holdout["persona_esperada"]) == set(classes)
assert set(train["puesto_texto"]).isdisjoint(set(holdout["puesto_texto"]))
print(f"Ejemplos etiquetados: {len(train)} train + {len(holdout)} holdout, por diseño humano")

model = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1, 2), lowercase=True, strip_accents="unicode")),
    ("classifier", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=42)),
])
model.fit(train["puesto_texto"], train["persona_esperada"])
holdout["prediccion"] = model.predict(holdout["puesto_texto"])
accuracy = accuracy_score(holdout["persona_esperada"], holdout["prediccion"])
report_raw = classification_report(
    holdout["persona_esperada"], holdout["prediccion"], labels=classes,
    output_dict=True, zero_division=0
)
report = pd.DataFrame.from_dict(
    {key: value for key, value in report_raw.items() if isinstance(value, dict)},
    orient="index"
)
print(f"Accuracy didáctica en holdout: {accuracy:.0%} ({len(holdout)} ejemplos)")
display(report.loc[classes, ["precision", "recall", "f1-score", "support"]].round(2))
fig, ax = plt.subplots(figsize=(7, 6))
ConfusionMatrixDisplay.from_predictions(
    holdout["persona_esperada"], holdout["prediccion"], labels=classes,
    display_labels=classes, xticks_rotation=30, cmap="Purples", ax=ax, colorbar=False
)
ax.set_title("Holdout sintético · no es precisión de producción")
fig.tight_layout()
display(fig)
plt.close(fig)
display(holdout[["puesto_texto", "persona_esperada", "prediccion"]].sort_values("persona_esperada"))

## Publicar la predicción como función SQL de sesión

Esta es una UDF de Python registrada para la sesión actual. No crea una función permanente en el catálogo.

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import StringType

def predict_persona(text):
    if text is None or not text.strip():
        return None
    # The fitted teaching model is small enough to serialize with the UDF.
    # Serverless notebooks do not expose SparkContext/broadcast variables.
    return str(model.predict([text])[0])

clasificar_persona_udf = F.udf(predict_persona, StringType())
spark.udf.register("clasificar_persona_ml", clasificar_persona_udf)
print("Función de sesión lista: clasificar_persona_ml(puesto_texto)")

spark.createDataFrame(
    [(str(title),) for title in holdout["puesto_texto"].tolist()], ["puesto_texto"]
).createOrReplaceTempView("titulos_holdout")
sql_preview = spark.sql("""
    SELECT puesto_texto, clasificar_persona_ml(puesto_texto) AS persona_predicha
    FROM titulos_holdout
    ORDER BY puesto_texto
""")
display(sql_preview)

## Aplicarlo a C2

Cuando el ETL de C2 ya haya creado `workspace.ai4data.dim_participante`, esta consulta clasifica como máximo 50 títulos y solo muestra conteos. Si el poll todavía no tiene respuestas o falta el ETL, la función queda demostrada con el holdout sintético de arriba.

In [ ]:
live_title_count = 0
try:
    catalog, schema, table = DIM.split(".")
    available = {r.tableName for r in spark.sql(f"SHOW TABLES IN {catalog}.{schema}").collect()}
    if table not in available:
        print("C2 todavía no tiene tabla modelada. Ejecuta: bash demos/clase-02/pipeline/etl_only.sh --dataset class2")
    else:
        live = (
            spark.table(DIM)
            .where(F.col("puesto_texto").isNotNull())
            .select("puesto_texto")
            .limit(50)
            .withColumn("persona_ml", F.expr("clasificar_persona_ml(puesto_texto)"))
        )
        live_rows = live.groupBy("persona_ml").count().orderBy(F.desc("count")).collect()
        live_title_count = sum(row["count"] for row in live_rows)
        if live_title_count:
            display(spark.createDataFrame(live_rows))
            print("Muestra limitada a 50; no se mostraron títulos ni identificadores.")
        else:
            print("C2 tiene 0 títulos todavía; la UDF se validó con el holdout etiquetado de arriba.")
except Exception as exc:
    if "TABLE_OR_VIEW_NOT_FOUND" in str(exc) or "SCHEMA_NOT_FOUND" in str(exc):
        print("C2 aún no está cargada; la demo SQL con el fixture sí quedó ejecutada arriba.")
    else:
        raise
print("Una etiqueta no es una decisión correcta por sí sola: necesitamos más ejemplos representativos y una persona responsable del modelo.")
dbutils.notebook.exit(
    f"ML_COMPLETE|train={len(train)}|holdout={len(holdout)}|accuracy={accuracy:.3f}|"
    f"live_titles={live_title_count}"
)